In [ ]:
# Environment bootstrap (works in both Colab and local)
try:
    import google.colab  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    print('[info] Running on Colab.')
else:
    print('[info] Running on local Jupyter.')


Mounted at /content/drive
[info] Running on Colab.


In [ ]:
import os, sys, pathlib
import numpy as np
import pandas as pd

if 'IS_COLAB' not in globals():
    try:
        import google.colab  # type: ignore
        IS_COLAB = True
    except Exception:
        IS_COLAB = False

if IS_COLAB:
    WORKDIR = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods')
else:
    cwd = pathlib.Path.cwd().resolve()
    if cwd.name == 'DML_methods':
        WORKDIR = cwd
    elif (cwd / 'DML_methods').is_dir():
        WORKDIR = (cwd / 'DML_methods').resolve()
    else:
        WORKDIR = cwd

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)

if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))

SUPP_DIR = WORKDIR / 'Supplement'
if SUPP_DIR.is_dir() and str(SUPP_DIR) not in sys.path:
    sys.path.insert(0, str(SUPP_DIR))

# Ensure 'Supplement' is a package if only estimation.py exists
init_path = SUPP_DIR / '__init__.py'
if SUPP_DIR.is_dir() and not init_path.exists():
    init_path.write_text('from .estimation import *\n', encoding='utf-8')
    print('[info] Created Supplement/__init__.py shim')

print(f'[info] WORKDIR: {WORKDIR}')


[info] WORKDIR: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods


In [ ]:
import importlib
import subprocess
import sys

def ensure_python_package(pkg_name: str) -> bool:
    try:
        importlib.import_module(pkg_name)
        return True
    except Exception:
        pass
    print(f'[info] Installing {pkg_name} ...')
    rc = subprocess.call([sys.executable, '-m', 'pip', 'install', '-q', pkg_name])
    return rc == 0

has_rpy2 = ensure_python_package('rpy2')
if has_rpy2:
    try:
        ip = get_ipython() if 'get_ipython' in globals() else None
        if ip is not None:
            ip.run_line_magic('load_ext', 'rpy2.ipython')
            print('[info] rpy2 extension loaded.')
        else:
            print('[info] rpy2 installed (not running inside IPython).')
    except Exception as e:
        print(f'[warn] Could not load rpy2.ipython extension: {e}')
else:
    print('[warn] rpy2 is not available. This notebook requires R + rpy2 + grf.')


[info] rpy2 extension loaded.


In [ ]:
try:
    import rpy2.robjects.packages as rpackages
    utils = rpackages.importr('utils')
    if not rpackages.isinstalled('grf'):
        utils.chooseCRANmirror(ind=1)
        utils.install_packages('grf')
        print("[info] Installed R package 'grf'.")
    else:
        print("[info] R package 'grf' already installed.")
except Exception as e:
    print(f"[warn] Could not verify/install R package 'grf': {e}")
    print("[warn] If local, install R and run: install.packages('grf')")


(as ‘lib’ is unspecified)










	‘/tmp/RtmpbQpfMd/downloaded_packages’



[info] Installed R package 'grf'.


In [ ]:
# -*- coding: utf-8 -*-
"""
Empirical Application: Job Corps (DDMLCT) — GRF version

What this script saves (to RESULTS_DIR):
1) estimates_GRF_seed{first}_to_seed{last}.csv
   - Stage-2 beta(t) estimates for every simulation seed
   - plus summary rows: mean, se

2) MISE_GRF_seed{first}_to_seed{last}.csv
   - MISE per seed (Stage-2 only)
   - plus summary rows: mean, se

(Optional)
3) estimates_stage1_GRF_seed{first}_to_seed{last}.csv
   - Stage-1 beta(t) estimates for every simulation seed
   - plus summary rows: mean, se
"""

import os
import sys
import pathlib
import logging
import numpy as np
import pandas as pd

# --- Environment Setup (Colab + local) ---
if "WORKDIR" in globals():
    BASE_DIR = pathlib.Path(WORKDIR).resolve()
else:
    try:
        import google.colab  # type: ignore
        BASE_DIR = pathlib.Path("/content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods")
    except Exception:
        cwd = pathlib.Path(".").resolve()
        if cwd.name == "DML_methods":
            BASE_DIR = cwd
        elif (cwd / "DML_methods").is_dir():
            BASE_DIR = (cwd / "DML_methods").resolve()
        else:
            BASE_DIR = cwd

# Add working directory and Supplement package to path
sys.path.append(str(BASE_DIR))
SUPP_DIR = BASE_DIR / "Supplement"
if SUPP_DIR.exists():
    sys.path.append(str(SUPP_DIR))

# Ensure Supplement package is importable
try:
    import Supplement
except ImportError:
    # Create __init__.py shim if missing
    if SUPP_DIR.exists() and not (SUPP_DIR / "__init__.py").exists():
        with open(SUPP_DIR / "__init__.py", "w") as f:
            f.write("from .estimation import *\n")
    import Supplement

# Import GRF models used in your original GRF pipeline
from Supplement.rgrf import regression_forest as RF_grf, regression_forest2 as RF2_grf

# Logging Setup
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger(__name__)


RESULTS_DIR = BASE_DIR / "Data_and_Results" / "Estimates"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Working Directory: {BASE_DIR}")
print(f"Results Directory: {RESULTS_DIR}")

# ===========================================================
# 1) Data Loading & Helpers
# ===========================================================
def load_jobcorps_data():
    emp_dir = BASE_DIR / "Data_and_Results"
    data_path = emp_dir / "emp_app.csv"
    semi_path = emp_dir / "semi-syn data grf.csv"
    h_star_path = emp_dir / "h_star_grf_empapp.csv"

    logger.info("Loading emp_app.csv...")
    data = pd.read_csv(data_path, index_col=0)

    # Consistent shuffling (fixed seed)
    data = data.sample(frac=1, random_state=20)

    # One-hot encoding
    data = pd.concat(
        [
            data.select_dtypes(exclude=["int64"]),
            pd.get_dummies(
                data.select_dtypes(include=["int64"]).astype("category"),
                drop_first=True,
                dtype=float,
            ),
        ],
        axis=1,
    )

    X = data.drop(["d", "y"], axis=1)
    T = data["d"]
    Y_emp = data["y"]

    logger.info("Loading semi-synthetic components...")
    semi_df = pd.read_csv(semi_path, index_col=0)
    if not np.array_equal(semi_df.index.values, data.index.values):
        semi_df = semi_df.loc[data.index]

    mu_hat = semi_df["mu_hat_grf"].to_numpy()
    g = semi_df["g_grf"].to_numpy()

    logger.info("Loading h_star ground truth...")
    h_star_df = pd.read_csv(h_star_path)

    # Keep t grid from file if available
    if "t" in h_star_df.columns:
        t_grid = h_star_df["t"].to_numpy()
    else:
        t_grid = np.arange(160, 2001, 40)

    h_star_vals = h_star_df["h_star"].to_numpy()

    return X, T, Y_emp, mu_hat, g, t_grid, h_star_vals


def gen_semi_y(mu_hat, g, rng):
    n = len(mu_hat)
    e = rng.choice([-1.0, 1.0], size=n)
    return mu_hat + e * g


def mise_against(est_beta, h_star_vals):
    return float(np.mean((np.asarray(est_beta) - np.asarray(h_star_vals)) ** 2))


def summarize_list(x_list):
    arr = np.asarray(x_list, dtype=float)
    mean = float(arr.mean()) if len(arr) > 0 else 0.0
    std = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    se = float(std / np.sqrt(len(arr))) if len(arr) > 0 else 0.0
    return mean, std, se


# ===========================================================
# 2) Main Simulation Function (GRF)
# ===========================================================
def run_grf_simulation(K_runs=100, base_seed=1):
    # Load data
    X, T, _, mu_hat, g, t_list, h_star_vals = load_jobcorps_data()
    n_obs = len(T)


    h_rule = np.std(T) * 3 * (n_obs ** (-0.2))
    h_first = 2 * h_rule
    L, u = 5, 0.5

    # GRF models
    model_rf1 = RF_grf()
    model_rf2 = RF2_grf()

    # Storage
    seeds = []
    stage1_beta = []
    stage2_beta = []
    stage2_mise = []

    logger.info(f"Starting {K_runs} simulations (GRF)...")

    for k in range(K_runs):
        seed = base_seed + k
        seeds.append(seed)

        print("", flush=True)
        print(f"[Progress] Processing Simulation {k+1}/{K_runs} (Seed: {seed})", flush=True)

        rng_sim = np.random.default_rng(seed)

        # Generate and shuffle
        Y_syn = gen_semi_y(mu_hat, g, rng_sim)
        perm = rng_sim.permutation(n_obs)

        X_k = X.iloc[perm].reset_index(drop=True)
        T_k = T.iloc[perm].reset_index(drop=True)
        Y_k = np.asarray(Y_syn[perm], float)

        # Stage 1: two bandwidth fits
        DDML_Class = Supplement.DDMLCT

        m1 = DDML_Class(model_rf1, model_rf2)
        m1.fit(X_k, T_k, Y_k, t_list, L, h=h_first, basis=False, standardize=True)

        m2 = DDML_Class(model_rf1, model_rf2)
        m2.fit(X_k, T_k, Y_k, t_list, L, h=h_first * u, basis=False, standardize=True)

        # Bandwidth selection
        Bt = (m1.beta - m2.beta) / ((m1.h ** 2) * (1 - (u ** 2)))
        h_star_ml = np.mean(((m2.Vt / (4 * (Bt ** 2))) ** 0.2) * (m1.n ** -0.2))

        # Stage-1 beta (at h_first)
        stage1_beta.append(np.asarray(m1.beta, dtype=float))

        # Stage 2: final fit with optimized bandwidth
        h_second = 0.8 * h_star_ml
        m_final = DDML_Class(model_rf1, model_rf2)
        m_final.fit(X_k, T_k, Y_k, t_list, L, h=h_second, basis=False, standardize=True)

        beta2 = np.asarray(m_final.beta, dtype=float)
        stage2_beta.append(beta2)

        mise2 = mise_against(beta2, h_star_vals)
        stage2_mise.append(mise2)

        if (k + 1) % 10 == 0:
            logger.info(f"Simulation {k+1}/{K_runs} completed.")

    # Print Stage-2 MISE summary
    mean2, std2, se2 = summarize_list(stage2_mise)
    print("\n" + "=" * 50)
    print(f"RESULTS (GRF, K={K_runs}) - Second Stage Only")
    print("=" * 50)
    print("Method: GRF")
    print("  [Stage 2: h = 0.8 * h_star_ml]")
    print(f"    Mean MISE : {mean2:.6f}")
    print(f"    Std MISE  : {std2:.6f}")
    print(f"    SE MISE   : {se2:.6f}")
    print("=" * 50 + "\n")

    return {
        "t_grid": np.asarray(t_list),
        "h_star": np.asarray(h_star_vals),
        "stage1_beta_mat": np.vstack(stage1_beta),
        "stage2_beta_mat": np.vstack(stage2_beta),
        "mise_list": stage2_mise,
        "seeds": seeds,
    }


# ===========================================================
# 3) Execution + Saving
# ===========================================================
K_SIM = 1
BASE_SEED = 1

results = run_grf_simulation(K_runs=K_SIM, base_seed=BASE_SEED)

first_seed = results["seeds"][0]
last_seed = results["seeds"][-1]

t_cols = [f"t_{int(t)}" for t in results["t_grid"]]

# ---- Save Stage-2 beta estimates  ----
df_beta2 = pd.DataFrame(results["stage2_beta_mat"], columns=t_cols)
df_beta2.insert(0, "seed", results["seeds"])

mean_row2 = pd.DataFrame([results["stage2_beta_mat"].mean(axis=0)], columns=t_cols)
mean_row2.insert(0, "seed", "mean")

se_row2 = pd.DataFrame(
    [results["stage2_beta_mat"].std(axis=0, ddof=1) / np.sqrt(len(results["seeds"]))],
    columns=t_cols,
)
se_row2.insert(0, "seed", "se")

df_beta2_final = pd.concat([df_beta2, mean_row2, se_row2], ignore_index=True)

out_beta2 = RESULTS_DIR / f"estimates_GRF_seed{first_seed}_to_seed{last_seed}.csv"
df_beta2_final.to_csv(out_beta2, index=False)
print(f"Saved Stage-2 beta estimates to: {out_beta2}")

# ---- Save MISE ----
df_mise = pd.DataFrame({"seed": results["seeds"], "mise": results["mise_list"]})

mise_mean, mise_std, mise_se = summarize_list(results["mise_list"])
df_mise_summary = pd.DataFrame(
    [
        {"seed": "mean", "mise": mise_mean},
        {"seed": "se", "mise": mise_se},
    ]
)

df_mise_final = pd.concat([df_mise, df_mise_summary], ignore_index=True)

out_mise = RESULTS_DIR / f"MISE_GRF_seed{first_seed}_to_seed{last_seed}.csv"
df_mise_final.to_csv(out_mise, index=False)
print(f"Saved MISE-only table to: {out_mise}")

logger.info("Execution finished.")


/usr/local/lib/python3.12/dist-packages/torch/__init__.py:1305: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at /pytorch/torch/csrc/tensor/python_tensor.cpp:434.)
  _C._set_default_tensor_type(t)


Working Directory: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods
Results Directory: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/Estimates

[Progress] Processing Simulation 1/1 (Seed: 1)


100%|██████████| 47/47 [3:58:06<00:00, 303.97s/it]
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:207: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(



RESULTS (GRF, K=1) - Second Stage Only
Method: GRF
  [Stage 2: h = 0.8 * h_star_ml]
    Mean MISE : 4.913084
    Std MISE  : 0.000000
    SE MISE   : 0.000000

Saved Stage-2 beta estimates to: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/Estimates/estimates_GRF_seed1_to_seed1.csv
Saved MISE-only table to: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/DML_methods/Data_and_Results/Estimates/MISE_GRF_seed1_to_seed1.csv
